# Forecast Visualization & Interactive Dashboards

This notebook creates comprehensive visualizations for 7-day forecasts and performance dashboards.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

from datetime import datetime, timedelta
from config import settings
from ingestion.cbk_stats import CBKMPesaClient
from models.ensemble import EnsembleForecaster

print("✓ Libraries loaded")

## 1. Load Data & Generate Forecast

In [ ]:
# Load data
cbk_client = CBKMPesaClient()
mpesa_data = cbk_client.get_mpesa_statistics()

series = pd.Series(
    mpesa_data['total_transactions'].values,
    index=pd.to_datetime(mpesa_data['date'])
)

# Train ensemble and generate forecast
ensemble = EnsembleForecaster()
ensemble.train(series, verbose=0)

forecast_result = ensemble.forecast(series, periods=7)

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'Date': forecast_result['dates'],
    'Forecast': forecast_result['ensemble'],
    'Lower_CI': forecast_result['lower'],
    'Upper_CI': forecast_result['upper'],
    'Prophet': forecast_result['prophet'],
    'LSTM': forecast_result['lstm']
})

print(f"📊 Data loaded: {len(series)} historical records")
print(f"📈 Forecast generated: {len(forecast_df)} days ahead")
print("\nForecast Summary:")
print(forecast_df.round(0))

## 2. Interactive 7-Day Forecast Chart

In [ ]:
# Create interactive forecast chart
fig = go.Figure()

# Historical data
fig.add_trace(go.Scatter(
    x=series.index[-60:],  # Last 60 days for context
    y=series.values[-60:],
    name='Historical',
    mode='lines',
    line=dict(color='#1f77b4', width=2),
    hovertemplate='<b>Historical</b><br>Date: %{x|%Y-%m-%d}<br>Transactions: %{y:,.0f}'
))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['Forecast'],
    name='Forecast',
    mode='lines+markers',
    line=dict(color='#ff7f0e', width=3),
    marker=dict(size=8, symbol='diamond'),
    hovertemplate='<b>Forecast</b><br>Date: %{x|%Y-%m-%d}<br>Transactions: %{y:,.0f}'
))

# Confidence interval (upper bound)
fig.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['Upper_CI'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

# Confidence interval (lower bound + fill)
fig.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['Lower_CI'],
    mode='lines',
    line=dict(width=0),
    fillcolor='rgba(255, 127, 14, 0.2)',
    fill='tonexty',
    name='95% Confidence Interval',
    hovertemplate='<b>CI Bounds</b><br>Lower: %{y:,.0f}'
))

fig.update_layout(
    title=dict(
        text='<b>7-Day M-Pesa Float Demand Forecast</b><br><sub>With 95% Confidence Interval</sub>',
        x=0.5,
        xanchor='center'
    ),
    xaxis_title='Date',
    yaxis_title='Daily Transactions',
    height=600,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255, 255, 255, 0.8)'),
    font=dict(size=12)
)

fig.show()

## 3. Model Contribution Analysis

# Create subplots showing individual model forecasts
fig = make_subplots(
    rows=1, cols=1,
    specs=[[{'secondary_y': False}]]
)

# Historical data
fig.add_trace(go.Scatter(
    x=series.index[-30:],
    y=series.values[-30:],
    name='Historical (30-day)',
    mode='lines',
    line=dict(color='#2ca02c', width=2),
    hovertemplate='Historical: %{y:,.0f}'
))

# Prophet forecast
fig.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['Prophet'],
    name='Prophet Forecast',
    mode='lines+markers',
    line=dict(color='#1f77b4', width=2, dash='dash'),
    marker=dict(size=7),
    hovertemplate='Prophet: %{y:,.0f}'
))

# LSTM forecast
fig.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['LSTM'],
    name='LSTM Forecast',
    mode='lines+markers',
    line=dict(color='#d62728', width=2, dash='dot'),
    marker=dict(size=7),
    hovertemplate='LSTM: %{y:,.0f}'
))

# Ensemble forecast
fig.add_trace(go.Scatter(
    x=forecast_df['Date'],
    y=forecast_df['Forecast'],
    name='Ensemble Forecast',
    mode='lines+markers',
    line=dict(color='#9467bd', width=3),
    marker=dict(size=10, symbol='star'),
    hovertemplate='Ensemble: %{y:,.0f}'
))

fig.update_layout(
    title='<b>Model Forecast Comparison</b>',
    xaxis_title='Date',
    yaxis_title='Daily Transactions',
    height=500,
    hovermode='x unified',
    template='plotly_white',
    font=dict(size=11)
)

fig.show()

## 4. Daily Breakdown

# Create detailed forecast table
detailed_forecast = forecast_df.copy()
detailed_forecast['Date'] = detailed_forecast['Date'].dt.strftime('%A, %Y-%m-%d')
detailed_forecast['Change_from_Prev'] = detailed_forecast['Forecast'].diff()
detailed_forecast['Change_Pct'] = detailed_forecast['Forecast'].pct_change() * 100
detailed_forecast['Confidence_Width'] = detailed_forecast['Upper_CI'] - detailed_forecast['Lower_CI']

# Create table figure
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Date</b>', '<b>Forecast</b>', '<b>Change</b>', '<b>Change %</b>', 
                '<b>Lower CI</b>', '<b>Upper CI</b>', '<b>CI Width</b>'],
        fill_color='#1f77b4',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[
            detailed_forecast['Date'],
            detailed_forecast['Forecast'].round(0).astype(str),
            detailed_forecast['Change_from_Prev'].round(0).astype(str),
            detailed_forecast['Change_Pct'].round(1).astype(str) + '%',
            detailed_forecast['Lower_CI'].round(0).astype(str),
            detailed_forecast['Upper_CI'].round(0).astype(str),
            detailed_forecast['Confidence_Width'].round(0).astype(str)
        ],
        fill_color='lavender',
        align='center',
        font=dict(size=11)
    )
)])

fig.update_layout(
    title='<b>7-Day Forecast Detail</b>',
    height=500
)

fig.show()

# Regional breakdown
regional_data = cbk_client.get_regional_summary()

# Create regional bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=regional_data['region'],
    y=regional_data['total_agents'],
    name='Total Agents',
    marker_color='#1f77b4'
))

fig.add_trace(go.Bar(
    x=regional_data['region'],
    y=regional_data['avg_daily_transactions'] / 100,
    name='Avg Daily Trans. (÷100)',
    marker_color='#ff7f0e'
))

fig.update_layout(
    title='<b>Regional Agent Distribution</b>',
    xaxis_title='Region',
    yaxis_title='Count',
    barmode='group',
    height=500,
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

## 5. Forecast Metrics Dashboard

# Calculate key metrics
forecast_stats = {
    'Average Daily Forecast': forecast_df['Forecast'].mean(),
    'Peak Forecast': forecast_df['Forecast'].max(),
    'Lowest Forecast': forecast_df['Forecast'].min(),
    'Std Dev': forecast_df['Forecast'].std(),
    'Avg Confidence Width': (forecast_df['Upper_CI'] - forecast_df['Lower_CI']).mean(),
    '7-Day Total': forecast_df['Forecast'].sum()
}

# Create metric cards
metrics_text = f"""
<b>FORECAST METRICS (7-Day Period)</b>

📊 Average Daily Forecast: {forecast_stats['Average Daily Forecast']:,.0f}
📈 Peak Forecast: {forecast_stats['Peak Forecast']:,.0f}
📉 Lowest Forecast: {forecast_stats['Lowest Forecast']:,.0f}
📐 Standard Deviation: {forecast_stats['Std Dev']:,.0f}
⚠️  Avg Confidence Width: {forecast_stats['Avg Confidence Width']:,.0f}
💰 7-Day Total: {forecast_stats['7-Day Total']:,.0f}
"""

print(metrics_text)

# Forecast quality indicators
print("\n✅ FORECAST QUALITY INDICATORS\n")

# Consistency check
daily_changes = forecast_df['Forecast'].diff().abs()
consistency = 100 * (1 - daily_changes.mean() / forecast_df['Forecast'].mean())
print(f"Forecast Consistency: {consistency:.1f}%")

# Confidence interval width
ci_width = (forecast_df['Upper_CI'] - forecast_df['Lower_CI']).mean()
forecast_mean = forecast_df['Forecast'].mean()
ci_pct = 100 * ci_width / forecast_mean
print(f"Avg CI Width: {ci_pct:.1f}% of forecast")

# Trend
trend = "📈 INCREASING" if forecast_df['Forecast'].iloc[-1] > forecast_df['Forecast'].iloc[0] else "📉 DECREASING"
trend_change = ((forecast_df['Forecast'].iloc[-1] - forecast_df['Forecast'].iloc[0]) / forecast_df['Forecast'].iloc[0]) * 100
print(f"Forecast Trend: {trend} ({trend_change:+.1f}%)")

# Model agreement
model_diff = (forecast_df['Prophet'] - forecast_df['LSTM']).abs().mean()
model_agreement = 100 * (1 - model_diff / forecast_df['Forecast'].mean())
print(f"Model Agreement: {model_agreement:.1f}%")

print("\n✅ Forecast Visualization Complete!")
print("\n📁 Output Locations:")
print(f"   CSV Export: {settings.OUTPUTS_DIR}/forecast_{datetime.now().strftime('%Y%m%d')}.csv")
print(f"   Database: float_forecasting.forecasts table")
print(f"   MLflow: {settings.MLFLOW_TRACKING_URI}")
print("\n🔄 Next Steps:")
print("   1. Save forecast to CSV")
print("   2. Upload to database")
print("   3. Notify stakeholders")
print("   4. Monitor actual performance")